# Hands-on 2 — Cosmology from Graphs: Regressing Ω_m

**Granada AI School — GNNs for Cosmology**

In Hands-on 1 each *halo* was a node and we classified nodes within one box. Now we go up one
level: **each simulation box is one graph**, and its label is the cosmological parameter **Ω_m**
that generated it. We train a GNN to read the graph and predict Ω_m — a hands-on version of
**simulation-based (likelihood-free) inference**.

You will:

1. Build a **dataset of many graphs** (one per box), each with a known Ω_m.
2. Use a **DataLoader** to mini-batch graphs.
3. Build a graph-level GNN with a **global pooling** readout.
4. Train with an MSE loss, then upgrade to a **moment network** that also predicts an
   **uncertainty** (mean + σ), and plot predicted-vs-true Ω_m with error bars.


## Step 0 — Setup

In [ ]:
!pip -q install torch torch-geometric scipy scikit-learn matplotlib pandas

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.metrics import r2_score

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 1 — A dataset of CAMELS-SAM boxes with different Ω_m

We use the **CAMELS-SAM LH** suite (`LH_0 … LH_999`), each a **100 Mpc/h** box with a different
cosmology, **Ω_m ∈ [0.1, 0.6]**. For each box we need its halo catalog and its Ω_m label
(`CosmoAstro_params.txt`, first number).

**How many boxes matters a lot.** This is *graph-level* regression: each box is one training
example. With ~20 boxes the model has ~14 training graphs and cannot learn (R² < 0). We therefore
use **100 boxes** (→ ~70 train), which is enough for a good fit.

**Keeping the download light — use a higher-redshift snapshot.** The z = 0 catalog is ~200 MB per
box (100 boxes ≈ 20 GB), because it contains *all* halos down to tiny masses. A **higher-redshift
snapshot is much smaller** (fewer, less-massive halos) — so 100 boxes becomes feasible:

| snapshot | redshift | size/box | 100 boxes |
|---|---|---|---|
| `out_99` | z = 0   | ~200 MB | ~20 GB |
| `out_20` | z ≈ 4.2 | ~95 MB  | ~9.5 GB |
| **`out_16`** | **z ≈ 5.2** | **~50 MB** | **~5 GB**  ← default |
| `out_12` | z ≈ 6.5 | ~19 MB  | ~1.9 GB |

> ⚠️ **Mass cut must match the snapshot.** At high z halos are far less massive — at z ≈ 5 the most
> massive halo is only ~10¹², so a 10¹² cut gives **zero** halos. We use `MASS_CUT = 3e10` for the
> default snapshot (→ a few thousand halos/box). If you change `SNAP`, revisit `MASS_CUT`.

Even at ~5 GB, downloading 100 boxes live per student is a lot: **pre-stage them once in a shared
Drive folder** (the cells skip files already present). Set `USE_REAL_DATA = False` for the built-in
synthetic generator (no download).

In [ ]:
import os, json, urllib.request
import numpy as np

USE_REAL_DATA = True
BOX_SIZE = 100.0        # Mpc/h  (CAMELS-SAM box)
SNAP     = 16           # z = 0 is out_99 (~200MB). out_16 ~ z=5.2 (~50MB) keeps 100 boxes light.
N_BOXES  = 100           # how many boxes to actually use (sampled across the Ω_m range)
N_POOL   = 500          # how many LH boxes to scan for Ω_m before selecting N_BOXES (raise for wider coverage)

BASE = "https://users.flatironinstitute.org/~camels/Rockstar/CAMELS-SAM/LH"

# --- where the data lives. On Colab this is your Google Drive (pre-stage here to skip downloads). ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/GNN_school/data_sam_lh'
except Exception:
    DATA_DIR = './data_sam_lh'          # local fallback
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

def param_url(i):   return f"{BASE}/LH_{i}/CosmoAstro_params.txt"
def catalog_url(i): return f"{BASE}/LH_{i}/Rockstar/out_{SNAP}.list"
def catalog_path(i):return os.path.join(DATA_DIR, f"CAMELS-SAM_LH{i}_out{SNAP}.list")

def read_omega_m(text):
    "First number in CosmoAstro_params.txt is Omega_m (2nd is sigma_8)."
    return float(text.split()[0])

# ---------- synthetic fallback (no download) ----------
def make_box(omega_m, box=BOX_SIZE, seed=0):
    rng = np.random.default_rng(seed)
    n_total = int(300 + 1400 * omega_m); f = min(0.2 + 1.2 * omega_m, 0.85)
    n_clu = int(f * n_total); n_field = n_total - n_clu
    n_centers = max(3, int(12 * omega_m)); sigma = max(1.2 - 1.2 * omega_m, 0.3)
    pts, centers = [], rng.uniform(0, box, size=(n_centers, 3)); per = max(1, n_clu // n_centers)
    for ctr in centers: pts.append(rng.normal(ctr, sigma, size=(per, 3)))
    pts.append(rng.uniform(0, box, size=(n_field, 3)))
    pos = np.concatenate(pts, axis=0) % box
    mass = 10 ** rng.normal(12.5, 0.4, size=len(pos))
    vel = np.abs(rng.normal(0, 100 + 500 * omega_m, size=len(pos)))  # |v|; dispersion grows with Ω_m
    return pos, mass, vel

def make_synthetic_dataset(n_boxes=300):
    rng = np.random.default_rng(42)
    omegas = rng.uniform(0.1, 0.6, size=n_boxes)
    return [(*make_box(om, seed=i), om) for i, om in enumerate(omegas)]

In [ ]:
# --- Select N_BOXES boxes spanning the Ω_m range, then download their catalogs ---
import pandas as pd

def scan_pool_omega(n_pool):
    "Read Omega_m for LH_0..LH_{n_pool-1}; cache to Drive so it is fetched only once."
    cache = os.path.join(DATA_DIR, f"omega_pool_{n_pool}.json")
    if os.path.exists(cache):
        return {int(k): v for k, v in json.load(open(cache)).items()}
    om = {}
    for i in range(n_pool):
        try:
            with urllib.request.urlopen(param_url(i), timeout=30) as r:
                om[i] = read_omega_m(r.read().decode())
        except Exception as e:
            print("skip LH_%d (%s)" % (i, e))
        if (i + 1) % 50 == 0:
            print("  scanned %d/%d parameter files" % (i + 1, n_pool))
    json.dump(om, open(cache, "w"))
    return om

def select_spanning(om, n):
    "Pick n boxes whose Omega_m values are spread evenly from min to max."
    items = sorted(om.items(), key=lambda kv: kv[1])          # (idx, Om) sorted by Om
    picks = np.linspace(0, len(items) - 1, n).round().astype(int)
    return [items[j] for j in picks]                          # list of (idx, Om)

def download_catalog(i):
    """Return (path, was_cached). If the file is already in DATA_DIR (your Drive) it is NOT
    re-downloaded. Otherwise it downloads to a temporary '.part' file and only renames it into
    place on success, so an interrupted download never leaves a partial file that looks complete."""
    path = catalog_path(i)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path, True                                     # already present -> skip download
    tmp = path + ".part"
    urllib.request.urlretrieve(catalog_url(i), tmp)
    os.replace(tmp, path)                                     # atomic: only a complete file lands at `path`
    return path, False

def load_box(path):
    df = pd.read_csv(path, sep=r'\s+', comment='#', header=None,
                     usecols=[2, 8, 9, 10, 11, 12, 13],
                     names=['Mvir', 'x', 'y', 'z', 'vx', 'vy', 'vz'])
    pos = df[['x', 'y', 'z']].to_numpy() % BOX_SIZE
    vel = np.linalg.norm(df[['vx', 'vy', 'vz']].to_numpy(), axis=1)   # peculiar-velocity magnitude
    return pos, df['Mvir'].to_numpy(), vel

if USE_REAL_DATA:
    om_pool  = scan_pool_omega(N_POOL)
    selected = select_spanning(om_pool, N_BOXES)              # [(idx, Om), ...]
    print("Selected %d boxes, Ω_m from %.3f to %.3f\n" %
          (len(selected), selected[0][1], selected[-1][1]))
    raw = []
    n_cached = n_new = 0
    for k, (idx, om) in enumerate(selected):
        path, cached = download_catalog(idx)
        n_cached += int(cached); n_new += int(not cached)
        pos, mass, vel = load_box(path)
        raw.append((pos, mass, vel, om))
        print("  [%3d/%d] LH_%-4d Ω_m=%.3f  halos=%7d  (%s)" %
              (k + 1, len(selected), idx, om, len(mass),
               "already in Drive - skipped" if cached else "downloaded"))
    print("\n%d already in Drive (not re-downloaded), %d newly downloaded." % (n_cached, n_new))
else:
    raw = make_synthetic_dataset()

print("\nboxes:", len(raw), "| Ω_m range: %.3f - %.3f" %
      (min(b[3] for b in raw), max(b[3] for b in raw)))

## Step 2 — Turn each box into a graph

Same recipe as Hands-on 1 (periodic radius graph, log-mass node feature), but now the label
`y` is a single number **per graph**: Ω_m.


In [ ]:
R_LINK   = 5.0            # Mpc/h linking radius
MASS_CUT = 3e10           # matched to the high-z default snapshot (out_16). Use ~1e12 for z=0.
MAX_HALOS_PER_BOX = 3000  # cap nodes per box for speed; None to use all halos above the cut

# Global normalisation stats (robust to whatever snapshot / mass cut you pick).
# Node features: log-mass and |velocity|.  Graph feature: log10(number of halos above the cut) —
# halo ABUNDANCE is one of the strongest Ω_m signals, and it must be fed explicitly because the
# node cap + mean-pool would otherwise hide it.
_lm = np.concatenate([np.log10(m[m >= MASS_CUT]) for _, m, _, _ in raw if (m >= MASS_CUT).any()])
_vv = np.concatenate([v[m >= MASS_CUT]           for _, m, v, _ in raw if (m >= MASS_CUT).any()])
_ln = np.log10([max(int((m >= MASS_CUT).sum()), 1) for _, m, _, _ in raw])
LOGM_MEAN, LOGM_STD = _lm.mean(), _lm.std() + 1e-6
VEL_MEAN,  VEL_STD  = _vv.mean(), _vv.std() + 1e-6
LOGN_MEAN, LOGN_STD = _ln.mean(), _ln.std() + 1e-6
print("nodes/box after cut: min %d, max %d | features: log-mass, |velocity| + halo-count" %
      (min(int((m >= MASS_CUT).sum()) for _, m, _, _ in raw),
       max(int((m >= MASS_CUT).sum()) for _, m, _, _ in raw)))

def box_to_graph(pos, mass, vel, omega_m, r=R_LINK, box=BOX_SIZE):
    keep = mass >= MASS_CUT
    pos, mass, vel = pos[keep], mass[keep], vel[keep]
    n_halos = len(mass)                                   # abundance BEFORE the speed cap
    if MAX_HALOS_PER_BOX is not None and len(pos) > MAX_HALOS_PER_BOX:
        s = np.random.choice(len(pos), MAX_HALOS_PER_BOX, replace=False)
        pos, mass, vel = pos[s], mass[s], vel[s]
    tree = cKDTree(pos, boxsize=box)
    pairs = tree.query_pairs(r, output_type="ndarray")
    if len(pairs) == 0:
        ei = torch.empty((2, 0), dtype=torch.long)
    else:
        ei = torch.tensor(np.concatenate([pairs.T, pairs.T[::-1]], axis=1), dtype=torch.long)
    x = torch.tensor(np.stack([(np.log10(mass) - LOGM_MEAN) / LOGM_STD,
                               (vel - VEL_MEAN) / VEL_STD], axis=1), dtype=torch.float32)
    d = Data(x=x, edge_index=ei, y=torch.tensor([omega_m], dtype=torch.float32))
    d.gfeat = torch.tensor([[(np.log10(max(n_halos, 1)) - LOGN_MEAN) / LOGN_STD]], dtype=torch.float32)
    return d

dataset = [box_to_graph(*b) for b in raw]
print(dataset[0], "| example Ω_m =", round(float(dataset[0].y), 3))

# sanity check: does connectivity correlate with Ω_m?  (it should)
mean_deg = [d.edge_index.shape[1] / max(d.num_nodes, 1) for d in dataset]
oms = [float(d.y) for d in dataset]
plt.scatter(oms, mean_deg, s=12); plt.xlabel(r"$\Omega_m$"); plt.ylabel("mean degree")
plt.title("Graph connectivity vs Ω_m"); plt.show()

## Step 3 — Train / val / test split and DataLoaders

In [ ]:
idx = np.random.permutation(len(dataset))
n_tr, n_va = int(0.7 * len(dataset)), int(0.15 * len(dataset))
train_set = [dataset[i] for i in idx[:n_tr]]
val_set   = [dataset[i] for i in idx[n_tr:n_tr+n_va]]
test_set  = [dataset[i] for i in idx[n_tr+n_va:]]

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=64)
test_loader  = DataLoader(test_set,  batch_size=64)
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)}")

## Step 4 — A graph-level GNN with global pooling

Two things make this work on real data:

- **Node features:** each halo carries its **log-mass** *and* its **|velocity|** — peculiar
  velocities scale with the growth rate (∝ Ω_m^0.55), a strong, direct Ω_m signal.
- **A graph-level halo-count feature:** after pooling we append `log(number of halos above the
  mass cut)`. Halo **abundance** is one of the classic cosmological probes, and the node cap +
  mean-pool would otherwise hide it.

The readout has two heads: a **mean** head (μ) and a **variance** head that reads a *detached*
embedding — so learning the uncertainty (for the error bars) never degrades the mean prediction.


In [ ]:
class GNNRegressor(torch.nn.Module):
    """Graph-level Ω_m regressor.
    Node features: log-mass, |velocity|.  A graph-level halo-COUNT feature is concatenated after
    pooling (abundance is a strong Ω_m signal). Two heads: a mean head (mu) and a variance head
    that reads a DETACHED embedding, so learning the uncertainty never degrades the mean."""
    def __init__(self, in_dim=2, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.mu_head  = torch.nn.Sequential(torch.nn.Linear(hidden + 1, hidden), torch.nn.ReLU(),
                                            torch.nn.Linear(hidden, 1))
        self.var_head = torch.nn.Sequential(torch.nn.Linear(hidden + 1, hidden), torch.nn.ReLU(),
                                            torch.nn.Linear(hidden, 1))

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.relu(self.conv2(x, data.edge_index))
        x = F.relu(self.conv3(x, data.edge_index))
        g = global_mean_pool(x, data.batch)            # [num_graphs, hidden]
        g = torch.cat([g, data.gfeat], dim=1)          # append halo-count feature
        mu      = self.mu_head(g)[:, 0]
        log_var = self.var_head(g.detach())[:, 0]      # detached -> variance can't corrupt the mean
        return mu, log_var

model = GNNRegressor(in_dim=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

def loss_fn(mu, log_var, target):
    # mean fit by clean MSE; variance fit to the (detached) residuals -> calibrated error bars
    mse = F.mse_loss(mu, target)
    var = 0.5 * torch.exp(-log_var) * (mu.detach() - target) ** 2 + 0.5 * log_var
    return mse + var.mean()

print(model)

## Step 5 — Train

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot, n = 0.0, 0
    torch.set_grad_enabled(train)
    for batch in loader:
        batch = batch.to(device)
        mu, log_var = model(batch)
        loss = loss_fn(mu, log_var, batch.y)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot += loss.item() * batch.num_graphs; n += batch.num_graphs
    torch.set_grad_enabled(True)
    return tot / n

tr_hist, va_hist = [], []
for epoch in range(1, 121):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    tr_hist.append(tr); va_hist.append(va)
    if epoch % 10 == 0:
        print(f"epoch {epoch:3d} | train NLL {tr:.3f} | val NLL {va:.3f}")

plt.plot(tr_hist, label="train"); plt.plot(va_hist, label="val")
plt.xlabel("epoch"); plt.ylabel("Gaussian NLL"); plt.legend(); plt.show()

## Step 6 — Evaluate: predicted vs true Ω_m

In [ ]:
model.eval()
mus, sigmas, trues = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        mu, log_var = model(batch)
        mus.append(mu.cpu().numpy())
        sigmas.append(torch.exp(0.5 * log_var).cpu().numpy())
        trues.append(batch.y.cpu().numpy())
mus, sigmas, trues = map(np.concatenate, (mus, sigmas, trues))

rmse = np.sqrt(np.mean((mus - trues) ** 2))
print(f"Test RMSE = {rmse:.4f} | R^2 = {r2_score(trues, mus):.3f}")

lims = [0.05, 0.62]
plt.figure(figsize=(6, 6))
plt.errorbar(trues, mus, yerr=sigmas, fmt="o", ms=4, alpha=0.6, ecolor="gray", capsize=2)
plt.plot(lims, lims, "k--", label="perfect")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel(r"true $\Omega_m$"); plt.ylabel(r"predicted $\Omega_m$")
plt.title(f"GNN inference of Ω_m  (RMSE={rmse:.3f})"); plt.legend(); plt.show()

## Step 7 — Experiments (your turn)

1. **Linking radius `r`.** Rebuild the dataset with `R_LINK` = 1.5, 3, 6 Mpc/h. Connectivity is
   the main signal here — how does it change the accuracy?
2. **Pooling.** `global_mean_pool` (used here) is stable. `global_add_pool` encodes the halo
   *count* (a real Ω_m signal) but summing over ~1000 nodes can blow up and give NaN losses —
   if you try it, normalise (e.g. divide by node count, or append `log(num_nodes)` as a graph
   feature) and lower the learning rate.
3. **Layers.** Replace `GCNConv` with `SAGEConv` or `EdgeConv`. Does a more expressive layer help?
4. **Two parameters.** If you have real CAMELS labels, make `y` two-dimensional `(Ω_m, σ8)` and
   predict both at once.
5. **Robustness.** Train on the synthetic set, test on real CAMELS (or vice-versa). GNNs that
   don't generalise across simulations are a live research topic (CAMELS robustness studies).


## Conclusion

Each box became a graph and the GNN learned to read a **cosmological parameter** off the
cosmic-web connectivity — with an uncertainty. This is the core idea behind using GNNs for
field-level, likelihood-free cosmological inference: no summary statistic (power spectrum,
2PCF) is computed by hand; the network extracts the information directly from the halo graph.
